In [3]:
import os
import json
import time
import hashlib
import argparse
from typing import List, Dict, Any, Optional, Tuple

import redis
from opensearchpy import OpenSearch
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
os.environ["OPENAI_API_KEY"]=""


# -----------------------
# CONFIG (match your ingestion)
# -----------------------
INDEX_NAME = os.getenv("OPENSEARCH_INDEX", "rag_pdf_index")
TEXT_FIELD = "text"
VECTOR_FIELD = "embedding"

EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")

# OpenSearch connection
OS_HOST = os.getenv("OPENSEARCH_HOST", "10.103.6.142")
OS_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
OS_USER = os.getenv("OPENSEARCH_USER", "admin")
OS_PASS = os.getenv("OPENSEARCH_PASS", "Aa1Bb2Cc3Dd4")

USE_SSL = os.getenv("OPENSEARCH_USE_SSL", "true").lower() == "true"
VERIFY_CERTS = os.getenv("OPENSEARCH_VERIFY_CERTS", "false").lower() == "true"
CA_CERTS = os.getenv("OPENSEARCH_CA_CERTS", "") or None
SSL_SHOW_WARN = os.getenv("OPENSEARCH_SSL_SHOW_WARN", "false").lower() == "true"

# Redis cache
REDIS_URL = os.getenv("REDIS_URL", "redis://10.103.6.142:6379/0")
REDIS_PREFIX = os.getenv("REDIS_PREFIX", "rag:os3")

# Cache TTLs (seconds)
TTL_EMBEDDING = int(os.getenv("TTL_EMBEDDING", "86400"))   # 1 day
TTL_RETRIEVAL = int(os.getenv("TTL_RETRIEVAL", "300"))     # 5 minutes
TTL_ANSWER = int(os.getenv("TTL_ANSWER", "300"))           # 5 minutes

# Toggle caches
CACHE_EMBEDDING = os.getenv("CACHE_EMBEDDING", "true").lower() == "true"
CACHE_RETRIEVAL = os.getenv("CACHE_RETRIEVAL", "true").lower() == "true"
#CACHE_ANSWER = os.getenv("CACHE_ANSWER", "false").lower() == "true"  # often safer off if docs change frequently
CACHE_ANSWER=True

# -----------------------
# Redis helpers
# -----------------------
def redis_client() -> redis.Redis:
    return redis.Redis.from_url(REDIS_URL, decode_responses=True)

def _stable_json(obj: Any) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False)

def _hash_key(*parts: Any) -> str:
    blob = "|".join(_stable_json(p) if isinstance(p, (dict, list, tuple)) else str(p) for p in parts)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()

def cache_get(r: redis.Redis, key: str) -> Optional[Any]:
    v = r.get(key)
    if v is None:
        return None
    try:
        return json.loads(v)
    except Exception:
        return None

def cache_set(r: redis.Redis, key: str, value: Any, ttl: int) -> None:
    r.setex(key, ttl, json.dumps(value, ensure_ascii=False))


# -----------------------
# OpenSearch client
# -----------------------
def connect_opensearch() -> OpenSearch:
    kwargs = dict(
        hosts=[{"host": OS_HOST, "port": OS_PORT}],
        http_auth=(OS_USER, OS_PASS),
        use_ssl=USE_SSL,
        verify_certs=VERIFY_CERTS,
        ssl_show_warn=SSL_SHOW_WARN,
        timeout=60,
        max_retries=3,
        retry_on_timeout=True,
    )
    if USE_SSL and VERIFY_CERTS and CA_CERTS:
        kwargs["ca_certs"] = CA_CERTS
    return OpenSearch(**kwargs)


# -----------------------
# Retrieval: BM25 + Vector
# -----------------------
def bm25_search(
    client: OpenSearch,
    query: str,
    k: int,
    filters: Optional[Dict[str, Any]] = None
) -> List[Dict[str, Any]]:
    must_filters = []
    if filters:
        for field, value in filters.items():
            must_filters.append({"term": {field: value}})

    body = {
        "size": k,
        "_source": [TEXT_FIELD, "source", "title", "chunk_index", "metadata"],
        "query": {
            "bool": {
                "must": [{"match": {TEXT_FIELD: {"query": query}}}],
                "filter": must_filters
            }
        }
    }
    resp = client.search(index=INDEX_NAME, body=body)
    return resp.get("hits", {}).get("hits", [])


def knn_search(
    client: OpenSearch,
    query_vector: List[float],
    k: int,
    filters: Optional[Dict[str, Any]] = None
) -> List[Dict[str, Any]]:
    must_filters = []
    if filters:
        for field, value in filters.items():
            must_filters.append({"term": {field: value}})

    body = {
        "size": k,
        "_source": [TEXT_FIELD, "source", "title", "chunk_index", "metadata"],
        "query": {
            "bool": {
                "filter": must_filters,
                "must": [
                    {
                        "knn": {
                            VECTOR_FIELD: {
                                "vector": query_vector,
                                "k": k
                            }
                        }
                    }
                ]
            }
        }
    }
    resp = client.search(index=INDEX_NAME, body=body)
    return resp.get("hits", {}).get("hits", [])


# -----------------------
# Fusion: RRF
# -----------------------
def rrf_fuse(
    bm25_hits: List[Dict[str, Any]],
    knn_hits: List[Dict[str, Any]],
    top_n: int,
    k_rrf: int = 60
) -> List[Tuple[Dict[str, Any], float]]:
    scores: Dict[str, Dict[str, Any]] = {}

    def add_hits(hits: List[Dict[str, Any]], weight: float = 1.0):
        for rank, h in enumerate(hits, start=1):
            doc_id = h["_id"]
            if doc_id not in scores:
                scores[doc_id] = {"hit": h, "score": 0.0}
            scores[doc_id]["score"] += weight * (1.0 / (k_rrf + rank))

    add_hits(bm25_hits, weight=1.0)
    add_hits(knn_hits, weight=1.0)

    fused = [(v["hit"], float(v["score"])) for v in scores.values()]
    fused.sort(key=lambda x: x[1], reverse=True)
    return fused[:top_n]


# -----------------------
# Context + LLM
# -----------------------
def build_context(fused_hits: List[Tuple[Dict[str, Any], float]], max_chars: int = 12000) -> str:
    parts = []
    total = 0
    for hit, score in fused_hits:
        src = hit.get("_source", {})
        text = (src.get(TEXT_FIELD) or "").strip()
        if not text:
            continue
        source_name = src.get("source") or (src.get("metadata", {}) or {}).get("source", "unknown")
        chunk_index = src.get("chunk_index", (src.get("metadata", {}) or {}).get("chunk_index", ""))
        header = f"[{source_name} | chunk {chunk_index} | rrf={score:.4f}]"
        block = header + "\n" + text + "\n"
        if total + len(block) > max_chars:
            break
        parts.append(block)
        total += len(block)
    return "\n---\n".join(parts)


def answer_with_openai(question: str, context: str) -> str:
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
    prompt = (
        "You are a precise assistant.\n"
        "Answer ONLY using the provided context.\n"
        "If the context is insufficient, say 'I don't know based on the provided documents.'\n\n"
        f"Question:\n{question}\n\n"
        f"Context:\n{context}\n"
    )
    return llm.invoke(prompt).content


# -----------------------
# Cached embedding
# -----------------------
def embed_query_cached(r: redis.Redis, embeddings: OpenAIEmbeddings, question: str) -> List[float]:
    cache_key = f"{REDIS_PREFIX}:emb:{_hash_key(EMBED_MODEL, question)}"
    if CACHE_EMBEDDING:
        cached = cache_get(r, cache_key)
        if isinstance(cached, list) and cached:
            return cached

    vec = embeddings.embed_query(question)
    if CACHE_EMBEDDING:
        cache_set(r, cache_key, vec, TTL_EMBEDDING)
    return vec


# -----------------------
# Cached retrieval
# -----------------------
def retrieve_hybrid_cached(
    r: redis.Redis,
    client: OpenSearch,
    question: str,
    qvec: List[float],
    bm25_k: int,
    knn_k: int,
    filters: Optional[Dict[str, Any]]
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    # Cache only stable, serializable parts
    # NOTE: We cache hits as returned by OpenSearch (includes _id, _score, _source), which is JSON-safe.
    base = {
        "index": INDEX_NAME,
        "question": question,
        "bm25_k": bm25_k,
        "knn_k": knn_k,
        "filters": filters or {},
    }
    bm25_key = f"{REDIS_PREFIX}:bm25:{_hash_key(base)}"
    knn_key = f"{REDIS_PREFIX}:knn:{_hash_key(base)}"

    if CACHE_RETRIEVAL:
        bm25_cached = cache_get(r, bm25_key)
        knn_cached = cache_get(r, knn_key)
        if isinstance(bm25_cached, list) and isinstance(knn_cached, list):
            return bm25_cached, knn_cached

    bm25_hits = bm25_search(client, question, bm25_k, filters=filters)
    knn_hits = knn_search(client, qvec, knn_k, filters=filters)

    if CACHE_RETRIEVAL:
        cache_set(r, bm25_key, bm25_hits, TTL_RETRIEVAL)
        cache_set(r, knn_key, knn_hits, TTL_RETRIEVAL)

    return bm25_hits, knn_hits


# -----------------------
# Main RAG
# -----------------------
def hybrid_rag_answer(
    question: str,
    bm25_k: int = 20,
    knn_k: int = 20,
    fuse_top_n: int = 8,
    filters: Optional[Dict[str, Any]] = None
) -> Dict[str, Any]:
    r = redis_client()
    client = connect_opensearch()

    # Optional: cache final answer (good for repeated identical questions)
    answer_key = f"{REDIS_PREFIX}:ans:{_hash_key({'q': question, 'bm25_k': bm25_k, 'knn_k': knn_k, 'top_n': fuse_top_n, 'filters': filters or {}})}"
    if CACHE_ANSWER:
        cached_ans = cache_get(r, answer_key)
        if isinstance(cached_ans, dict) and "answer" in cached_ans:
            cached_ans["cached"] = True
            return cached_ans

    embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
    qvec = embed_query_cached(r, embeddings, question)

    bm25_hits, knn_hits = retrieve_hybrid_cached(
        r=r,
        client=client,
        question=question,
        qvec=qvec,
        bm25_k=bm25_k,
        knn_k=knn_k,
        filters=filters
    )

    fused = rrf_fuse(bm25_hits, knn_hits, top_n=fuse_top_n, k_rrf=60)
    context = build_context(fused)
    answer = answer_with_openai(question, context)

    sources = []
    for hit, score in fused:
        src = hit.get("_source", {}) or {}
        sources.append({
            "id": hit.get("_id"),
            "source": src.get("source") or (src.get("metadata", {}) or {}).get("source"),
            "chunk_index": src.get("chunk_index", (src.get("metadata", {}) or {}).get("chunk_index")),
            "rrf_score": score,
        })

    out = {
        "answer": answer,
        "sources": sources,
        "bm25_count": len(bm25_hits),
        "knn_count": len(knn_hits),
        "cached": False,
    }

    if CACHE_ANSWER:
        cache_set(r, answer_key, out, TTL_ANSWER)

    return out


# -----------------------
# CLI
# -----------------------
# if __name__ == "__main__":
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--q", required=True, help="Question to ask")
#     parser.add_argument("--bm25_k", type=int, default=20)
#     parser.add_argument("--knn_k", type=int, default=20)
#     parser.add_argument("--top_n", type=int, default=8)
#     parser.add_argument("--filter_source", default=None, help="Optional: filter by source filename (keyword field: source)")
#     args = parser.parse_args()

#     filters = None
#     if args.filter_source:
#         filters = {"source": args.filter_source}

#     out = hybrid_rag_answer(
#         args.q,
#         bm25_k=args.bm25_k,
#         knn_k=args.knn_k,
#         fuse_top_n=args.top_n,
#         filters=filters,
#     )

#     print("\nANSWER:\n", out["answer"])
#     print("\nCACHED:", out["cached"])
#     print("\nSOURCES:")
#     for s in out["sources"]:
#         print(s)

In [4]:
out = hybrid_rag_answer(
        "Explain what are states in this state machine implementation",
        bm25_k=20,
        knn_k=20,
        fuse_top_n=8,
        filters=None,
    )
print(out)

{'answer': '- **RELEVANCE**  \n  - *Gatekeeping: determine domain relevance; may terminate early if irrelevant.*\n\n- **CONFIDENCE**  \n  - *Score answerability via strict JSON; also generate a fast-draft snippet for reference.*\n\n- **DECOMPOSITION**  \n  - *Break question into 2–3 concrete subtopics (device physics / system impact / implementation).*\n\n- **SELF EVAL**  \n  - *Per-subtopic confidence (discrete {0, .25, .5, .75, 1 }); decides whether to search online or proceed.*\n\n- **SEARCH ONLINE**  \n  - *Optional: retrieve 1–k items per low-confidence.*', 'sources': [{'id': 'I4fzt5wBoHDILo4XwdFE', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 64, 'rrf_score': 0.032018442622950824}, {'id': '7Ifzt5wBoHDILo4XwdBD', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 9, 'rrf_score': 0.03125763125763126}, {'id': 'BIfzt5wBoHDILo4XwdFD', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 33, 'rrf_score': 0.031054405392392875}, {'id': '94fzt5wBoHDILo4XwdBD', 'source': 'RA_FSM_Paper 2.pdf', 'chu

In [13]:
out = hybrid_rag_answer(
        "Explain what are states in this state machine implementation",
        bm25_k=20,
        knn_k=20,
        fuse_top_n=8,
        filters=None,
    )
print(out)

{'answer': '- **RELEVANCE**  \n  - *Gatekeeping: determine domain relevance; may terminate early if irrelevant.*\n\n- **CONFIDENCE**  \n  - *Score answerability via strict JSON; also generate a fast-draft snippet for reference.*\n\n- **DECOMPOSITION**  \n  - *Break question into 2–3 concrete subtopics (device physics / system impact / implementation).*\n\n- **SELF EVAL**  \n  - *Per-subtopic confidence (discrete {0, .25, .5, .75, 1 }); decides whether to search online or proceed.*\n\n- **SEARCH ONLINE**  \n  - *Optional: retrieve 1–k items per low-confidence.*', 'sources': [{'id': 'I4fzt5wBoHDILo4XwdFE', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 64, 'rrf_score': 0.032018442622950824}, {'id': '7Ifzt5wBoHDILo4XwdBD', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 9, 'rrf_score': 0.03125763125763126}, {'id': 'BIfzt5wBoHDILo4XwdFD', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 33, 'rrf_score': 0.031054405392392875}, {'id': '94fzt5wBoHDILo4XwdBD', 'source': 'RA_FSM_Paper 2.pdf', 'chu

In [6]:
out = hybrid_rag_answer(
        "Explain state machine implementation",
        bm25_k=20,
        knn_k=20,
        fuse_top_n=8,
        filters=None,
    )
print(out)

{'answer': '- **Finite-State Machine (FSM) Control**  \n  - *FSMs are used to control deterministic behavior in various applications, including dialogue systems and robotic task planning. They provide a structured approach to managing states and transitions based on specific conditions, allowing for confidence-driven, iterative refinement in processes like query handling.*\n\n- **State Transitions and Responsibilities**  \n  - *The FSM operates through defined states such as Relevance, Confidence, Decomposition, Self-Evaluation, and Answer. Each state has specific responsibilities and exit conditions that dictate the flow of the process, ensuring that the system can adaptively refine its responses based on confidence levels and the relevance of the information retrieved.*', 'sources': [{'id': 'I4fzt5wBoHDILo4XwdFE', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 64, 'rrf_score': 0.032266458495966696}, {'id': '7Ifzt5wBoHDILo4XwdBD', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 9, 'rrf

In [7]:
out = hybrid_rag_answer(
        "Explain state machine implementation",
        bm25_k=20,
        knn_k=20,
        fuse_top_n=8,
        filters=None,
    )
print(out)

{'answer': '- **Finite-State Machine (FSM) Control**  \n  - *FSMs are used to control deterministic behavior in various applications, including dialogue systems and robotic task planning. They provide a structured approach to managing states and transitions based on specific conditions, allowing for confidence-driven, iterative refinement in processes like query handling.*\n\n- **State Transitions and Responsibilities**  \n  - *The FSM operates through defined states such as Relevance, Confidence, Decomposition, Self-Evaluation, and Answer. Each state has specific responsibilities and exit conditions that dictate the flow of the process, ensuring that the system can adaptively refine its responses based on confidence levels and the relevance of the information retrieved.*', 'sources': [{'id': 'I4fzt5wBoHDILo4XwdFE', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 64, 'rrf_score': 0.032266458495966696}, {'id': '7Ifzt5wBoHDILo4XwdBD', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 9, 'rrf

In [10]:
out = hybrid_rag_answer(
        "Explain state machine implementation",
        bm25_k=20,
        knn_k=20,
        fuse_top_n=8,
        filters=None,
    )
print(out)

{'answer': '- **Finite-State Machine (FSM) Control**  \n  - *FSMs are used to govern query processing in a structured manner, transitioning through states such as Idle, Relevance, Confidence, Decomposition, Self-Evaluation, and Answer. This control allows for a systematic approach to handling user queries and managing confidence levels in the responses.*\n\n- **State Responsibilities**  \n  - *Each state in the FSM has specific responsibilities, such as determining relevance, assessing confidence, decomposing questions into subtopics, and composing final answers. This structured design helps in maintaining clarity and efficiency in the retrieval and answering process.*', 'sources': [{'id': 'I4fzt5wBoHDILo4XwdFE', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 64, 'rrf_score': 0.032266458495966696}, {'id': '7Ifzt5wBoHDILo4XwdBD', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 9, 'rrf_score': 0.0315136476426799}, {'id': '94fzt5wBoHDILo4XwdBD', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index

In [11]:
out = hybrid_rag_answer(
        "Explain what are states in this state machine implementation",
        bm25_k=20,
        knn_k=20,
        fuse_top_n=8,
        filters=None,
    )
print(out)

{'answer': '- **RELEVANCE**  \n  - *Gatekeeping: determine domain relevance; may terminate early if irrelevant.*\n\n- **CONFIDENCE**  \n  - *Score answerability via strict JSON; also generate a fast-draft snippet for reference.*\n\n- **DECOMPOSITION**  \n  - *Break question into 2–3 concrete subtopics (device physics / system impact / implementation).*\n\n- **SELF EVAL**  \n  - *Per-subtopic confidence (discrete {0, .25, .5, .75, 1 }); decides whether to search online or proceed.*\n\n- **SEARCH ONLINE**  \n  - *Optional: retrieve 1–k items per low-confidence.*', 'sources': [{'id': 'I4fzt5wBoHDILo4XwdFE', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 64, 'rrf_score': 0.032018442622950824}, {'id': '7Ifzt5wBoHDILo4XwdBD', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 9, 'rrf_score': 0.03125763125763126}, {'id': 'BIfzt5wBoHDILo4XwdFD', 'source': 'RA_FSM_Paper 2.pdf', 'chunk_index': 33, 'rrf_score': 0.031054405392392875}, {'id': '94fzt5wBoHDILo4XwdBD', 'source': 'RA_FSM_Paper 2.pdf', 'chu